# Session 01: train a model, save it, serve it
Run this notebook from `intro/`, the copy of `sessions/01_intro/` made in the session brief, with its Python 3.12 environment active. Run the cells from top to bottom. The exercise is a regression; HighRev stays the course's classification case study.

In [ ]:
from pathlib import Path
import json
import platform
import importlib.metadata as metadata
import joblib
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score

ROOT = Path.cwd()
if not (ROOT / "Dockerfile").exists():
    raise RuntimeError("Start Jupyter from intro/, the directory that holds the Dockerfile.")
print("Python:", platform.python_version())
for package in ["pandas", "scikit-learn", "joblib"]:
    print(package, metadata.version(package))


## Load data and identify the target
This small dataset comes with scikit-learn; no separate download or account is required. Use raw feature values (`scaled=False`) so preprocessing is fitted on training rows. The target is a quantitative measure of disease progression after one year. This is an educational example, not a clinical service.


In [ ]:
dataset = load_diabetes(as_frame=True, scaled=False)
X = dataset.data
Y = dataset.target
print("Rows:", len(X), "Features:", list(X.columns))
display(X.head())
print(dataset.DESCR)


## Fit and evaluate a simple regression
Keep 20% of rows aside for evaluation. A split is part of model evaluation, not an automated software test. Compare Ridge with predicting the training mean. Lower MAE is better; R² compares performance with a constant prediction. No metric threshold is required for this introduction.


In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, Y, test_size=0.2, random_state=42
)
model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=1.0))
model.fit(X_train, y_train)
mean_model = DummyRegressor(strategy="mean").fit(X_train, y_train)
for name, fitted in [("Training mean", mean_model), ("Ridge", model)]:
    predicted = fitted.predict(X_valid)
    print(name, {"MAE": mean_absolute_error(y_valid, predicted),
                 "R2": r2_score(y_valid, predicted)})


## Save the model and one request
Save the complete preprocessing pipeline, feature names and target together. The API will load this fitted artifact; it will not train on startup. Only load artifacts you created or trust. The request contains input features, never the target.


In [ ]:
artifacts = ROOT / "artifacts"
artifacts.mkdir(exist_ok=True)
bundle = {"model": model, "feature_names": list(X.columns),
          "target": "disease_progression_after_one_year"}
joblib.dump(bundle, artifacts / "model.joblib")
row = X_valid.iloc[0]
request = {"features": {name: float(row[name]) for name in X.columns}}
(ROOT / "example.json").write_text(json.dumps(request, indent=2))
local_prediction = float(model.predict(X_valid.iloc[[0]])[0])
(ROOT / "expected.json").write_text(json.dumps({"prediction": local_prediction}, indent=2))
print("Saved:", artifacts / "model.joblib")
print("Local prediction:", local_prediction)
print(json.dumps(request, indent=2))


## Continue in the session brief
Record the metrics and versions, then restart the kernel and run all cells to check that the notebook does not depend on hidden state. Back in the terminal: freeze the environment, start the local API, build the Docker image, and send `example.json` to `/predict`. Compare the returned number with `expected.json`. No test suite yet; that is session 3.